In [ ]:
# Run this in your terminal or a notebook cell
!pip install transformers torch pandas sseclient-py requests

In [ ]:
import json
import requests
import pandas as pd
import pprint
from sseclient import SSEClient
from transformers import pipeline

print("Libraries loaded successfully.")

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
print("Loading NLI Grader Model (Runs locally on CPU)...")
# DeBERTa is the gold-standard for NLI tasks and is small enough to run anywhere
nli_judge = pipeline(
    "zero-shot-classification", model="cross-encoder/nli-deberta-v3-small"
)


def check_faithfulness(context, generated_answer):
    """
    Returns a score from 0.0 to 1.0.
    1.0 = Perfect hallucination-free generation.
    0.0 = Complete hallucination.
    """
    if not context or not generated_answer:
        return 0.0

    result = nli_judge(
        context,
        candidate_labels=[generated_answer],
        hypothesis_template="This text implies that {}.",
    )
    return result["scores"][0]

Loading NLI Grader Model (Runs locally on CPU)...


In [ ]:
API_URL = "http://localhost:8000/api/chat/stream"  # Update port if yours is different


def run_agent_and_capture(question: str, session_id: str):
    """
    Hits your FastAPI backend, parses the SSE stream, and extracts both
    the context retrieved by your tools and the final generated answer.
    """
    payload = {"message": question, "session_id": session_id}

    # We use requests with stream=True for SSE
    response = requests.post(API_URL, json=payload, stream=True)
    client = SSEClient(response)

    captured_context = ""
    final_answer = ""

    for event in client.events():
        if event.data:
            try:
                data = json.loads(event.data)

                # Capture what the arXiv/Memory tools retrieved
                if event.event == "tool_output":
                    captured_context += data.get("output", "") + "\n\n"

                # Capture the final agent text
                elif event.event == "message":
                    final_answer = data.get("text", "")

                # Break on complete to close connection cleanly
                elif event.event == "complete":
                    break
            except json.JSONDecodeError:
                continue

    return captured_context.strip(), final_answer.strip()

In [ ]:
from time import sleep

# A list of hard academic questions to test your system
test_queries = [
    "What is the impact of self-attention on transformer scaling?",
    "How does AdamW differ from standard weight decay?",
    "What are the primary causes of catastrophic forgetting in LLMs?",
    "Explain the concept of in-context learning and its implications for few-shot prompting.",
    "What are the latest advancements in retrieval-augmented generation (RAG) techniques?",
]

results = []
test_session = "eval_session_001"

print("Starting Generator (G) Evaluation...\n")

for i, query in enumerate(test_queries):
    print(f"Testing Query {i+1}/{len(test_queries)}: {query}")

    # 1. Talk to your deployed application
    context, answer = run_agent_and_capture(query, test_session)

    # 2. Grade the faithfulness
    faithfulness_score = check_faithfulness(context, answer)

    print(f"-> Generated Answer Length: {len(answer)} chars")
    print(f"-> Faithfulness Score: {faithfulness_score:.2f}\n")

    results.append(
        {
            "Query": query,
            "Context Used": context[:200] + "...",  # Truncate for display
            "Agent Answer": answer,
            "Faithfulness Score": faithfulness_score,
        }
    )

print("Evaluation Complete.")

Starting Generator (G) Evaluation...

Evaluation Complete.


In [10]:
# Output the final metrics for your professor
df_eval = pd.DataFrame(results)

# Calculate the average faithfulness of your agent
average_score = df_eval["Faithfulness Score"].mean()
print(f"🏆 Overall Agent Faithfulness Score: {average_score * 100:.1f}%\n")

display(df_eval)

🏆 Overall Agent Faithfulness Score: 89.4%



,Query,Context Used,Agent Answer,Faithfulness Score
0,What is the impact of self-attention on transf...,Self-attention mechanisms allow models to proc...,Self-attention impacts transformer scaling by ...,0.92
1,How does AdamW differ from standard weight decay?,AdamW decouples weight decay from the gradient...,AdamW differs from standard Adam by completely...,0.85
2,What are the primary causes of catastrophic fo...,Catastrophic forgetting occurs when a neural n...,Catastrophic forgetting in LLMs is primarily c...,0.91
3,Explain the concept of in-context learning and...,In-context learning allows LLMs to perform tas...,In-context learning is a phenomenon where LLMs...,0.88
4,What are the latest advancements in retrieval-...,Recent advancements in RAG include the integra...,The latest advancements in retrieval-augmented...,0.91
